# RL - Chapter 4 - DP - Grid Problem


In [9]:
from toc import generate_toc

path = 'ch_4_DP_p1_grid_problem.ipynb'
generate_toc(path)

## Table of Contents

- [RL - Chapter 4 - DP - Grid Problem](#rl-chapter-4-dp-grid-problem)
  - [Grid Problem - Environment](#grid-problem-environment)
  - [Policy](#policy)
  - [Dynamic Programming Agent](#dynamic-programming-agent)
  - [Run Policy Iteration](#run-policy-iteration)

In [10]:
import numpy as np
import matplotlib.pyplot as plt

# set decimal precision for numpy arrays
np.set_printoptions(precision=1, suppress=True)

## Grid Problem - Environment


In [11]:
class GridWorldEnv:
    def __init__(self):
        # define grid world parameters
        self.n_rows = 4
        self.n_cols = 4

        # define state space
        n_states = self.n_rows * self.n_cols
        self.states = np.arange(n_states)
        self.n_states = n_states
        self.goal_states = [0, n_states - 1]        # goal states

        # define action space
        self.n_actions = 4                          # number of actions
        self.actions = np.arange(self.n_actions)    # actions: 0=up, 1=down, 2=left, 3=right

        # define transition probabilities
        # P[s] = {a : (p(s'|s,a), s', r), ...}
        self.P: dict = None

        # build transition probabilities
        self.build_transition_probabilities()

    def build_transition_probabilities(self):
        # P[s] = {a : (p(s'|s,a), s', r), ...}
        self.P = {s: {a: None for a in self.actions} for s in self.states}

        for s in self.states:
            row, col = divmod(s, self.n_cols)
            if s in self.goal_states:
                continue
            for a in self.actions:
                if a == 0:  # up
                    next_row, next_col = max(row - 1, 0), col
                elif a == 1:  # down
                    next_row, next_col = min(row + 1, self.n_rows - 1), col
                elif a == 2:  # left
                    next_row, next_col = row, max(col - 1, 0)
                elif a == 3:  # right
                    next_row, next_col = row, min(col + 1, self.n_cols - 1)
                else:
                    raise ValueError("Invalid action")

                next_s = next_row * self.n_cols + next_col
                reward = -1  # if next_s not in self.goal_states else 0
                self.P[s][a] = (1, next_s, reward)

## Policy

In this implementation, we use action probabilities. \
All actions have a probability.\
Another approach is to set a single action as a policy, which is a deterministic policy.


In [12]:
# define Policy

class Policy:
    def __init__(self, env: GridWorldEnv):
        # pi[s, a] = p(a|s)
        self.pi = np.zeros((env.n_states, env.n_actions))
        self.initialize_uniform_policy()

    def initialize_uniform_policy(self):
        self.pi[:] = 1.0 / self.pi.shape[1]

## Dynamic Programming Agent


In [13]:
# Dynamic Programming Agent
class DP_Agent:
    def __init__(self, env: GridWorldEnv):

        # initialize environment(dynamics) and policy
        self.env = env              # Environment
        self.policy = Policy(env)   # Policy

        # setting hyperparameters
        self.gamma = 1.0            # Discount factor
        self.theta = 1e-3           # Convergence threshold

        # initialize value function
        self.V = np.zeros(env.n_states)

        # counters for diagnostics
        self.n_policy_eval = 0
        self.n_policy_imp = 0

    def policy_iteration(self):
        while True:
            self.policy_evaluation()
            policy_stable = self.policy_improvement()
            self.n_policy_imp += 1
            if policy_stable:
                break

    def policy_evaluation(self):
        while True:
            delta = 0.0
            for s in self.env.states:
                if s in self.env.goal_states:
                    continue

                # expected_return
                action_values = self.get_action_values(state=s, pi_s=self.policy.pi[s])
                v_new = np.sum(action_values)

                # update value function and track max change for convergence check
                delta = max(delta, abs(self.V[s] - v_new))
                self.V[s] = v_new

            self.n_policy_eval += 1
            if delta < self.theta:
                break

    def policy_improvement(self):
        policy_stable = True
        threshold = 1e-4

        for s in self.env.states:
            if s in self.env.goal_states:
                continue

            # update policy
            old_p_actions = self.policy.pi[s].copy()
            new_p_actions = self.get_improved_action_probabilities(s)
            self.policy.pi[s] = new_p_actions

            # check if policy is stable
            if not np.allclose(old_p_actions, new_p_actions, atol=threshold):
                policy_stable = False

        return policy_stable

    def get_action_values(self, state, pi_s=None):
        # action_values[a] = q(s,a) * p(a|s) for all actions a in state s
        if pi_s is None:
            pi_s = np.ones(self.env.n_actions)
        action_values = np.zeros(self.env.n_actions)
        for a in self.env.actions:
            prob, next_s, reward = self.env.P[state][a]
            action_values[a] = pi_s[a] * prob * (reward + self.gamma * self.V[next_s])
        return action_values

    def get_improved_action_probabilities(self, state):
        # q_s: q(s,a) for all actions a in state s
        q_s = self.get_action_values(state)
        best_q = np.max(q_s)
        n_best = np.sum(q_s == best_q)
        p_actions = np.zeros_like(q_s)
        p_actions[q_s == best_q] = 1.0 / n_best
        return p_actions

## Run Policy Iteration


In [14]:
# define environment and physics
env = GridWorldEnv()

# run policy evaluation for Grid World environment
agent = DP_Agent(env)

# evaluate the policy
agent.policy_evaluation()

# print state-value function
print("State-value function under the random policy:")
print(agent.V.reshape(env.n_rows, env.n_cols))
print(f"Evaluation sweeps: {agent.n_policy_eval}")

State-value function under the random policy:
[[  0. -14. -20. -22.]
 [-14. -18. -20. -20.]
 [-20. -20. -18. -14.]
 [-22. -20. -14.   0.]]
Evaluation sweeps: 88


In [15]:
# policy iteration
agent.policy_iteration()

# print state-value function
print("State-value function after policy iteration:")
print(agent.V.reshape(env.n_rows, env.n_cols))
print(f"Policy improvement steps: {agent.n_policy_imp}")
print(f"Total evaluation sweeps: {agent.n_policy_eval}")

State-value function after policy iteration:
[[ 0. -1. -2. -3.]
 [-1. -2. -3. -2.]
 [-2. -3. -2. -1.]
 [-3. -2. -1.  0.]]
Policy improvement steps: 3
Total evaluation sweeps: 93
